# Strategy 14. Naive strategy with graph schema

Evaluating strategy 14 - naive approach with graph schema - on BioMix test-set. Using enhanced schema.


In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]


In [3]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")


    
from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [5]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

Loading from an extended biomix test-set

In [6]:
questions = pd.read_csv("../biomix/testset/biomix_true_false_selected_augmented.csv")

## Running template-based query

- 14b - enchanced schema


In [7]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph.refresh_schema()

normal_schema = graph.schema

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_83954/506362744.py:5: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {nam

Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `targetInModel`: STRING 
  - `targetInModelMgiId`: STRING 
  - `targetFromSourceId`: STRING 
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `name`: STRING Example: "pathological process"
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `

In [8]:
from langchain_core.prompts import PromptTemplate

system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

{schema}

"""


normal_schema_description = f"""\
This is graph schema:
--------------------------------------------
{normal_schema}
--------------------------------------------    
"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{enhanced_schema}
--------------------------------------------    
"""

system_prompt_normal_schema = system_prompt_generic.format(schema = normal_schema_description)
system_prompt_enhanced_schema = system_prompt_generic.format(schema = enhanced_schema_description)

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)

# Option 14b - enhanced schema

In [9]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

# models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
models = ["gpt-4o", "gpt-5", "gpt-5.2-2025-12-11", "claude-sonnet-4-20250514", "claude-sonnet-4-5-20250929"]
niter = 1
todo = [(m,r['text']) for _,r in questions.iterrows() for m in models for _ in range(niter)]

def run_llm_14b(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        messages = [
            SystemMessage(content=system_prompt_enhanced_schema),
            HumanMessage(content=user_prompt.text)
        ]
        result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_14b(llm_model, question))


Prompting LLM: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [2:28:46<00:00, 17.85s/it]


In [10]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_14b(llm_model, question)
        time.sleep(2)

In [11]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:  36%|███████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                                                                                                       | 181/500 [04:25<13:42,  2.58s/it]Transaction failed and will be retried in 1.0862871976298467s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Transaction failed and will be retried in 0.8868466639833953s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Transaction failed and will be retried in 1.033636421390999s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Querying graph:  72%|███████████

In [13]:
results = process_results(todo, llm_answers, cypher_results)
with open("../RIW_190_NL_to_KG_benchmarking/results/biomix14-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

In [14]:
results_df = pd.DataFrame(results)
results_df["has_interaction"] = results_df["count"] > 0
results_df = results_df.merge(questions, left_on="question", right_on="text", how="left")
results_df = results_df.drop(columns=["text"])
results_df['direct'] = ~results_df['question'].str.contains("is not associated")
results_df["answer"] = results_df["direct"] == results_df["has_interaction"]

In [15]:
results_df.to_excel("../RIW_190_NL_to_KG_benchmarking/results/biomix14-evaluations.xlsx", index=False)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error,has_interaction,label,direct,answer
0,gpt-4o,Polycythemia Vera is not associated with Gene ...,To determine whether there is an association b...,[{'query': 'MATCH (d:DiseaseOrPhenotypicFeatur...,1,"MATCH (d:DiseaseOrPhenotypicFeature {name: ""Po...",True,[],0.344416,0.0,NaN,False,False,False,True
1,gpt-5,Polycythemia Vera is not associated with Gene ...,MATCH (g)\nWHERE (g:HumanGene OR g:Gene) AND g...,[],0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True
2,gpt-5.2-2025-12-11,Polycythemia Vera is not associated with Gene ...,```cypher\n// Check whether there is ANY GeneT...,[{'query': '// Check whether there is ANY Gene...,1,// Check whether there is ANY GeneToDiseaseAss...,False,NaN,NaN,NaN,Function execution timed out,False,False,False,True
3,claude-sonnet-4-20250514,Polycythemia Vera is not associated with Gene ...,I'll help you create a Cypher query to find as...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,"[{'gene_symbol': 'JAK2', 'gene_id': 'ensembl:E...",0.439718,518.0,NaN,True,False,False,False
4,claude-sonnet-4-5-20250929,Polycythemia Vera is not associated with Gene ...,I'll help you create a Cypher query to verify ...,[{'query': '// Check for any association betwe...,2,// Check for any association between Polycythe...,True,"[{'Disease': 'polycythemia vera', 'DiseaseID':...",0.157719,25.0,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,gpt-4o,Smith-Lemli-Opitz Syndrome is associated with ...,To find the association between Smith-Lemli-Op...,"[{'query': 'MATCH (d:Disease {name: ""Smith-Lem...",1,"MATCH (d:Disease {name: ""Smith-Lemli-Opitz Syn...",True,[],0.110916,0.0,NaN,False,False,True,False
496,gpt-5,Smith-Lemli-Opitz Syndrome is associated with ...,MATCH (g)\nWHERE (g:HumanGene OR g:Gene) AND g...,[],0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True,False
497,gpt-5.2-2025-12-11,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\n// Find evidence that TBX5 is assoc...,[{'query': '// Find evidence that TBX5 is asso...,1,// Find evidence that TBX5 is associated with ...,True,[],0.131550,0.0,NaN,False,False,True,False
498,claude-sonnet-4-20250514,Smith-Lemli-Opitz Syndrome is associated with ...,I'll help you create a Cypher query to find th...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.114257,0.0,NaN,False,False,True,False


In [19]:
# Calculate the fraction of correct answers for each model
accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()
accuracy_df.columns = ['model', 'accuracy']
accuracy_df

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_83954/2029851061.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()


,model,accuracy
0,claude-sonnet-4-20250514,0.84
1,claude-sonnet-4-5-20250929,0.49
2,gpt-4o,0.58
3,gpt-5,0.50
4,gpt-5.2-2025-12-11,0.87


In [1]:
accuracy_df

NameError: name 'accuracy_df' is not defined